# SemanticPrism Demonstration Workflow
A clean, step-by-step interactive workflow breaking down the pipeline execution.
Correctly ordered to demonstrate True Raw Extractions -> Normalization -> Topology -> Synthesis.


In [1]:
import os
import json
import glob
import time
from datetime import datetime
import logging
import IPython.display
import nest_asyncio
nest_asyncio.apply()

from typing import Any
from src.core.logger import save_execution_log
from src.extraction.extractor import ExtractionPipeline
from src.extraction.normalize_text import execute_normalization_phase
from src.embedding.embedding import EmbeddingPipeline
from src.nlp.hypernyms import HypernymPipeline
from src.nlp.nlp_mapping import NamingResolutionPipeline
from src.topology.graph_builder import TopologyEngine
from src.synthesis.synthesizer import SynthesisEngine
from src.helpers.visualizer import SemanticVisualizer

def _save_state(data: Any, filepath: str):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        default_encoder = lambda x: list(x) if isinstance(x, set) else str(x)
        if isinstance(data, list) and len(data) > 0 and hasattr(data[0], 'model_dump'):
            json.dump([item.model_dump(mode='json') for item in data], f, indent=4, default=default_encoder)
        elif hasattr(data, 'model_dump'):
            json.dump(data.model_dump(mode='json'), f, indent=4, default=default_encoder)
        else:
            json.dump(data, f, indent=4, default=default_encoder)

target_dir = "inputs/testdocs"
files = glob.glob(os.path.join(target_dir, "*.txt")) + glob.glob(os.path.join(target_dir, "*.md"))
raw_texts = []
for path in files:
    with open(path, "r", encoding="utf-8") as f:
        raw_texts.append(f.read())
print(f"Successfully loaded {len(raw_texts)} documents natively from {target_dir}.")
if not raw_texts:
    print("WARNING: Create physical text files manually inside inputs/testdocs gracefully.")


Successfully loaded 3 documents natively from inputs/testdocs.


### 1. Extraction Pipeline (Raw Extraction)

In [2]:
extractor = ExtractionPipeline("config.yaml")

start_time = time.time()
start_datetime = datetime.now()
workflow_errors = []

# Baseline trackings 
original_subjs, original_preds, original_objs = set(), set(), set()
norm_subjs, norm_preds, norm_objs = set(), set(), set()

all_themes = []
for idx, text in enumerate(raw_texts):
    print(f"Processing themes for document {idx + 1}/{len(raw_texts)}")
    themes = await extractor.discover_themes(text)
    all_themes.extend(themes)
_save_state(all_themes, "outputs/01_extraction/original_themes.json")

weighted_string = extractor.weight_themes(all_themes)
master_context = await extractor.consolidate_themes(weighted_string)
_save_state(master_context, "outputs/01_extraction/distilled_themes.json")

master_domain = master_context.master_domain if master_context else "General"
raw_triples = []
for idx, text in enumerate(raw_texts):
    print(f"Processing triples for document {idx + 1}/{len(raw_texts)}")
    trips = await extractor.extract_triples(text, master_context)
    raw_triples.extend(trips)
_save_state(raw_triples, "outputs/01_extraction/original_triplets.json")

print(f"Logically Extracted Triples seamlessly across all documents: {len(raw_triples)}")

original_subjs = {t.subject for t in raw_triples}
original_preds = {t.predicate for t in raw_triples}
original_objs = {t.object for t in raw_triples}

visualizer = SemanticVisualizer()
visualizer.visualize_triples(raw_triples, "outputs/01_extraction/01_raw_triples_graph.html", "Phase 1: Raw Extractions")
IPython.display.display(IPython.display.IFrame("outputs/01_extraction/01_raw_triples_graph.html", width="100%", height="600px"))


2026-04-20 20:31:53 | INFO     | ContextManager | Initialized ContextManager - VRAM Free: 11860 MB, RAM Free: 22972 MB
2026-04-20 20:31:53 | INFO     | ExtractionPipeline | Initialized ExtractionPipeline (Model: mistral-nemo:12b-instruct-2407-q4_K_M, Use Async: True)
Processing themes for document 1/3
2026-04-20 20:31:53 | INFO     | ExtractionPipeline | Phase 1: Executing Theme Discovery...
Processing themes for document 2/3
2026-04-20 20:32:16 | INFO     | ExtractionPipeline | Phase 1: Executing Theme Discovery...
Processing themes for document 3/3
2026-04-20 20:32:36 | INFO     | ExtractionPipeline | Phase 1: Executing Theme Discovery...
2026-04-20 20:33:01 | INFO     | ExtractionPipeline | Phase 1.5: Consolidating overarching Master Domain.
Processing triples for document 1/3
2026-04-20 20:33:08 | INFO     | ExtractionPipeline | Phase 2: Executing Logical Triple Extraction Explicitly...
2026-04-20 20:33:08 | INFO     | ExtractionPipeline | Extracting triples from logic chunk 1/1...

### 1.5. Text Normalization
This phase safely normalizes text strings in-place within the raw_triples array.

In [3]:
normalized_triples, norm_subjs, norm_preds, norm_objs = await execute_normalization_phase(
    extractor,
    raw_triples,
    master_domain,
    _save_state
)
print("Text Normalization Complete.")
visualizer = SemanticVisualizer()
visualizer.visualize_triples(normalized_triples, "outputs/01_extraction/01_normalized_triples_graph.html", "Phase 1: Normalized Extractions")
IPython.display.display(IPython.display.IFrame("outputs/01_extraction/01_normalized_triples_graph.html", width="100%", height="600px"))

2026-04-20 20:36:05 | INFO     | NormalizationPhase | ==================================================
2026-04-20 20:36:05 | INFO     | NormalizationPhase | STAGE 2.5: LLM TEXT NORMALIZATION
2026-04-20 20:36:05 | INFO     | NormalizationPhase | ==================================================
2026-04-20 20:36:05 | INFO     | NormalizationPhase | Executing NLP Preprocessing standardizations natively...
2026-04-20 20:36:05 | INFO     | NormalizationPhase | Normalizing Subject String Batch 1/1
2026-04-20 20:36:05 | INFO     | ExtractionPipeline | Phase 2.5: Executing Strict Lexical Normalization
2026-04-20 20:36:15 | INFO     | NormalizationPhase | Normalizing Predicate String Batch 1/1
2026-04-20 20:36:15 | INFO     | ExtractionPipeline | Phase 2.5: Executing Strict Lexical Normalization
2026-04-20 20:36:52 | INFO     | NormalizationPhase | Normalizing Object String Batch 1/1
2026-04-20 20:36:52 | INFO     | ExtractionPipeline | Phase 2.5: Executing Strict Lexical Normalization
Text 

### 2. Embedding Compression

In [4]:
embedder = EmbeddingPipeline("config.yaml")

proposed_clusters = embedder.process_triples(normalized_triples)
_save_state(proposed_clusters, "outputs/02_embedding/clusters_identified.json")

print(f"Mathematical Compression Output: {len(proposed_clusters)} distinct physical clusters grouped by Euclidean cosine thresholds.")


2026-04-20 20:37:38 | INFO     | EmbeddingPipeline | Initializing Offline Embedding Pipeline (Model: all-MiniLM-L6-v2)
2026-04-20 20:37:38 | INFO     | EmbeddingPipeline | Loading embedding model locally from: models/embeddings/all-MiniLM-L6-v2


This model was created with Sentence Transformers version 5.4.1, but you're using version 5.4.0. Consider updating to the latest version to avoid potential issues.


2026-04-20 20:37:38 | INFO     | EmbeddingPipeline | Executing Offline Embedding Matrix completely accurately intelligently.
2026-04-20 20:37:38 | INFO     | EmbeddingPipeline | Grouping 'subject' logic array accurately cleanly dependably gracefully.
2026-04-20 20:37:38 | INFO     | EmbeddingPipeline | Processing vector mappings rigidly explicitly cleanly.
2026-04-20 20:37:38 | INFO     | EmbeddingPipeline | Generating vectors. Unique items: 6
2026-04-20 20:37:39 | INFO     | EmbeddingPipeline | Dynamic Eigengap Analysis identified optimal components: 1/35
2026-04-20 20:37:39 | INFO     | EmbeddingPipeline | PCA reduced logically gracefully. Dimensions elegantly: (6, 1)
2026-04-20 20:37:39 | INFO     | EmbeddingPipeline | Grouping 'object' logic array accurately cleanly dependably gracefully.
2026-04-20 20:37:39 | INFO     | EmbeddingPipeline | Processing vector mappings rigidly explicitly cleanly.
2026-04-20 20:37:39 | INFO     | EmbeddingPipeline | Generating vectors. Unique items: 3

### 3. Taxonomic Hypernym Lifting (Hybrid)

In [5]:
hypernyms = HypernymPipeline("config.yaml")

verified_clusters = await hypernyms.validate_context_vectors(proposed_clusters, master_domain)
_save_state(verified_clusters, "outputs/03_hypernym_lifting/verified_clusters.json")

hypernym_mapping = await hypernyms.taxonomic_lift(verified_clusters, master_domain)
_save_state(hypernym_mapping, "outputs/03_hypernym_lifting/hypernym_mapping.json")

print("Hybrid LLM Verification and taxonomic mapping extracted explicitly.")


2026-04-20 20:37:39 | INFO     | HypernymPipeline | Initializing Master LLM Factory for context lifting.
2026-04-20 20:37:39 | INFO     | ContextManager | Initialized ContextManager - VRAM Free: 11538 MB, RAM Free: 22167 MB
2026-04-20 20:37:39 | INFO     | HypernymPipeline | Initializing Dedicated Embedding Encoder for centroid projections.
2026-04-20 20:37:39 | INFO     | HypernymPipeline | Loading embedding model locally from: models/embeddings/all-MiniLM-L6-v2


This model was created with Sentence Transformers version 5.4.1, but you're using version 5.4.0. Consider updating to the latest version to avoid potential issues.


2026-04-20 20:37:39 | INFO     | HypernymPipeline | Phase 3.1: Validating mathematical proposals...
2026-04-20 20:37:39 | INFO     | HypernymPipeline | Evaluating 2 mathematical clusters for [subject]
2026-04-20 20:37:43 | INFO     | HypernymPipeline | Evaluating 0 mathematical clusters for [predicate]
2026-04-20 20:37:43 | INFO     | HypernymPipeline | Evaluating 31 mathematical clusters for [object]
2026-04-20 20:37:47 | INFO     | HypernymPipeline | Phase 3.2: Native Taxonomic Lifting.
Hybrid LLM Verification and taxonomic mapping extracted explicitly.


### 4. Taxonomic Naming Resolution

In [6]:
mapper = NamingResolutionPipeline()

mapped_triples = mapper.resolve_names(normalized_triples, hypernym_mapping)
_save_state(mapped_triples, "outputs/04_mapping/mapped_triplets.json")

visualizer.visualize_triples(mapped_triples, "outputs/04_mapping/02_resolved_triples_graph.html", "Phase 4: Abstracted Taxonomy")
IPython.display.display(IPython.display.IFrame("outputs/04_mapping/02_resolved_triples_graph.html", width="100%", height="600px"))


2026-04-20 20:37:47 | INFO     | NamingResolution | Initializing Taxonomic Resolution & Consolidation Pipeline.
2026-04-20 20:37:47 | INFO     | NamingResolution | Resolving taxonomy natively for 35 semantic triples.
2026-04-20 20:37:47 | INFO     | NamingResolution | Resolution cleanly completed safely. Triples formally updated: 35
2026-04-20 20:37:47 | INFO     | Visualizer | Generating abstract visual organically seamlessly correctly optimally for 35 logical facts securely.
2026-04-20 20:37:47 | INFO     | Visualizer | Visual flawlessly written to: outputs/04_mapping/02_resolved_triples_graph.html


### 5. NetworkX Topological Community Graphing

In [7]:
topology = TopologyEngine()

graph = topology.build_graph(mapped_triples)
partition = topology.detect_communities(graph)
hierarchy = topology.extract_hierarchy(graph, partition)

_save_state(partition, "outputs/05_topology/modularity_partition.json")
_save_state(hierarchy, "outputs/05_topology/extracted_hierarchy.json")

visualizer.visualize_topology(graph, partition, "outputs/05_topology/03_topology_communities_graph.html", "Phase 5: Modularity Topology")
IPython.display.display(IPython.display.IFrame("outputs/05_topology/03_topology_communities_graph.html", width="100%", height="600px"))


2026-04-20 20:37:47 | INFO     | TopologyEngine | Initializing TopologyEngine statically.
2026-04-20 20:37:47 | INFO     | TopologyEngine | Synthesizing structured framework securely from 35 formal semantic triples.
2026-04-20 20:37:47 | INFO     | TopologyEngine | Topological Map completely mapped explicitly structurally natively. Nodes: 37, Edges: 34
2026-04-20 20:37:47 | INFO     | TopologyEngine | Computing Leiden best_partition securely mathematically safely...
2026-04-20 20:37:47 | INFO     | TopologyEngine | Resolution identified securely. Unique functional subsets: 4
2026-04-20 20:37:47 | INFO     | TopologyEngine | Computing explicit global topological relations formally naturally natively.
2026-04-20 20:37:47 | INFO     | Visualizer | Generating geometric topology abstraction logically natively safely.
2026-04-20 20:37:47 | INFO     | Visualizer | Structural Geometric Community visually exported to: outputs/05_topology/03_topology_communities_graph.html


### 6. Semantic JSON Structure Synthesis

In [8]:
synthesizer = SynthesisEngine("config.yaml")
resolved_schemas = await synthesizer.generate_schemas(hierarchy, master_domain)
file_path = synthesizer.build_global_context(resolved_schemas)

print(f"Final output synthesized seamlessly to: {file_path}")


2026-04-20 20:37:47 | INFO     | SynthesisEngine | Initializing Master LLM Factory for semantic synthesis mappings natively.
2026-04-20 20:37:47 | INFO     | ContextManager | Initialized ContextManager - VRAM Free: 11476 MB, RAM Free: 21959 MB
2026-04-20 20:37:47 | INFO     | SynthesisEngine | Synthesis Engine Initialized natively.
2026-04-20 20:37:47 | INFO     | SynthesisEngine | Synthesizing 4 mapped communities dynamically.
2026-04-20 20:40:20 | INFO     | SynthesisEngine | Executing Cross-Community Output Parsing safely.
2026-04-20 20:40:20 | INFO     | SynthesisEngine | Pydantic Python Schemas brilliantly dynamically structurally successfully mapped to outputs/semantic_models.py
2026-04-20 20:40:20 | INFO     | SynthesisEngine | Master Synthesis exported flawlessly to outputs/semantic_prism_master_graph.json
Final output synthesized seamlessly to: outputs/semantic_prism_master_graph.json


### 7. Logging & Diagnostic Preservation

In [9]:
end_time = time.time()
duration = end_time - start_time

all_errors = workflow_errors.copy()
if hasattr(extractor, 'llm'): all_errors.extend(extractor.llm.error_history)
if hasattr(hypernyms, 'llm'): all_errors.extend(hypernyms.llm.error_history)
if hasattr(synthesizer, 'llm'): all_errors.extend(synthesizer.llm.error_history)

all_ctxs = []
if hasattr(extractor, 'llm'): all_ctxs.extend(extractor.llm.context_history)
if hasattr(hypernyms, 'llm'): all_ctxs.extend(hypernyms.llm.context_history)
if hasattr(synthesizer, 'llm'): all_ctxs.extend(synthesizer.llm.context_history)

distilled_t_count = len(master_context.master_themes) if master_context and hasattr(master_context, 'master_themes') else 0
metrics = {
    "start_datetime": start_datetime,
    "duration": duration,
    "use_async": getattr(extractor, 'use_async', False),
    "model_name": extractor.config.get('llm', {}).get('model_name', 'Unknown'),
    "connection_protocol": extractor.config.get('llm', {}).get('connection_protocol', 'Unknown'),
    "doc_count": len(raw_texts),
    "doc_lengths": [len(doc) for doc in raw_texts],
    "all_ctxs": all_ctxs,
    "all_themes_count": len(all_themes),
    "distilled_t_count": distilled_t_count,
    "raw_triples_count": len(raw_triples),
    "orig_subjs": len(original_subjs),
    "orig_preds": len(original_preds),
    "orig_objs": len(original_objs),
    "norm_subjs": len(norm_subjs),
    "norm_preds": len(norm_preds),
    "norm_objs": len(norm_objs),
    "all_errors": all_errors
}

workflow_logger = logging.getLogger("DiagnosticWorkflow")
save_execution_log(metrics, workflow_logger)
print("Diagnostics preserved successfully.")


Diagnostics preserved successfully.
